In [ ]:
import pandas as pd
import glob
import os

# 1. 読み込みたいファイルが保存されているパスを指定（例: 'results/*.csv'）
# ディレクトリ名やファイルパターンのルールに合わせて書き換えてください
file_path = 'CIFAR10-Subsets/result2-*.csv'

# 2. 条件にマッチするファイルパスのリストを取得
files = glob.glob(file_path)
print(f"読み込み対象ファイル数: {len(files)}")

# 3. リスト内包表記で各CSVを読み込み、一つのデータフレームに結合
# ignore_index=True を指定すると、結合後にインデックスを 0 から振り直します
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

# 4. 結果の確認
print(f"結合後のデータ形状: {df.shape}")
display(df.head())

In [ ]:
df.columns

In [ ]:
df[df[' detectThreshold'] == 0.6]

In [ ]:
# ラベル単位での正答数をカウント
pairs = [
    [' 0', 'Label_0', 'Label_0_count', 'Label_0_total', 'Label_0_unknown', 'Label_0_unknown_Acc', 'Label_0_Acc'],
    ['1', 'Label_1', 'Label_1_count', 'Label_1_total', 'Label_1_unknown', 'Label_1_unknown_Acc', 'Label_1_Acc'],
    ['2', 'Label_2', 'Label_2_count', 'Label_2_total', 'Label_2_unknown', 'Label_2_unknown_Acc', 'Label_2_Acc'],
    ['3', 'Label_3', 'Label_3_count', 'Label_3_total', 'Label_3_unknown', 'Label_3_unknown_Acc', 'Label_3_Acc'],
    ['4', 'Label_4', 'Label_4_count', 'Label_4_total', 'Label_4_unknown', 'Label_4_unknown_Acc', 'Label_4_Acc'],
    ['5', 'Label_5', 'Label_5_count', 'Label_5_total', 'Label_5_unknown', 'Label_5_unknown_Acc', 'Label_5_Acc'],
    ['6', 'Label_6', 'Label_6_count', 'Label_6_total', 'Label_6_unknown', 'Label_6_unknown_Acc', 'Label_6_Acc'],
    ['7', 'Label_7', 'Label_7_count', 'Label_7_total', 'Label_7_unknown', 'Label_7_unknown_Acc', 'Label_7_Acc'],
    ['8', 'Label_8', 'Label_8_count', 'Label_8_total', 'Label_8_unknown', 'Label_8_unknown_Acc', 'Label_8_Acc'],
    ['9', 'Label_9', 'Label_9_count', 'Label_9_total', 'Label_9_unknown', 'Label_9_unknown_Acc', 'Label_9_Acc'],
]

#summaryDF = pd.DataFrame()
#summaryDF = pd.DataFrame(columns=['accuracy'])

indexes = []
addData = {}

ks = df[' k'].unique()
dbThresholds = df[' dbThreshold'].unique()
detectThresholds = df[' detectThreshold'].unique();
refines = df['refine'].unique()

line = 0

for skipLabel in [0,1,2,3,4,5,6,7,8,9,99]:

    if not 'skipLabel' in addData:
        addData['skipLabel'] = []

    for k in ks:
        if not 'k' in addData:
            addData['k'] = []
    
        for dbThreshold in dbThresholds:
    
            if not 'dbThreshold' in addData:
                addData['dbThreshold'] = []
    
            for detectThreshold in detectThresholds:
    
                if not 'detectThreshold' in addData:
                    addData['detectThreshold'] = []
    
                for refine in refines:
        
                    if not 'refine' in addData:
                        addData['refine'] = []
                
                    addData['skipLabel'].append(int(skipLabel))
                    addData['k'].append(int(k))
                    addData['dbThreshold'].append(float(dbThreshold))
                    addData['detectThreshold'].append(float(detectThreshold))
                    addData['refine'].append(str(refine))
                
                    totalCorrect = 0
                    totalSum = 0
                    totalUnknown = 0
    
                    for [label1, label2, countLabel, totalLabel, unknownLabel, unknownAccLabel, accLabel] in pairs:
                        df_tmp = df[(df[' k']==k) & (df[' dbThreshold'] == dbThreshold) & (df[' detectThreshold'] == detectThreshold) & (df['refine'] == refine) & (df[' skipLabel'] == skipLabel)]                    
                        correct = df_tmp[(df_tmp[' predictLabels'] == label2)][[label1]].sum().sum()
                        #print(correct)
                        total = df_tmp[(df_tmp[' predictLabels'] == label2)][[' 0','1', '2', '3', '4', '5', '6', '7', '8', '9', ' Unknown']].sum().sum()
                        unknown = df_tmp[(df_tmp[' predictLabels'] == label2)][[' Unknown']].sum().sum()
                        #print(total)
                        #print(correct/total, total, unknown, unknown/total)
                
                        if not label2 in addData:
                            addData[label2] = []
    
                        if total == 0:
                            addData[label2].append(float(0))                        
                        else:
                            addData[label2].append(float(correct/total))
    
                        if not countLabel in addData:
                            addData[countLabel] = []
                        if not totalLabel in addData:
                            addData[totalLabel] = []
                        if not unknownLabel in addData:
                            addData[unknownLabel] = []
                        if not unknownAccLabel in addData:
                            addData[unknownAccLabel] = []
                        if not accLabel in addData:
                            addData[accLabel] = []
                        addData[countLabel].append(correct)
                        addData[totalLabel].append(total)
                        addData[unknownLabel].append(unknown)
                        if total == 0:
                            addData[unknownAccLabel].append(0)
                            addData[accLabel].append(0)
                        else:
                            addData[unknownAccLabel].append(unknown/total)
                            addData[accLabel].append(correct/total)
                        
                        totalCorrect += correct
                        totalSum += total
                        totalUnknown += unknown
                
                    if not 'totalAcc' in addData:
                        addData['totalAcc'] = []
                    if not 'unknownRate' in addData:
                        addData['unknownRate'] = []
                    if not 'incorrectRate' in addData:
                        addData['incorrectRate'] = []
                    if not 'knownAcc' in addData:
                        addData['knownAcc'] = []
                    if not 'totalCorrect' in addData:
                        addData['totalCorrect'] = []
                    if not 'totalSum' in addData:
                        addData['totalSum'] = []
                    if not 'totalUnknown' in addData:
                        addData['totalUnknown'] = []
                    if not 'totalMiss' in addData:
                        addData['totalMiss'] = []
    
                    if totalSum == 0:
                        addData['totalAcc'].append(float(0))
                        addData['unknownRate'].append(float(0))
                        addData['incorrectRate'].append(float(0))
                        addData['knownAcc'].append(float(0))
                        addData['totalCorrect'].append(float(0))
                        addData['totalUnknown'].append(float(0))
                        addData['totalSum'].append(float(0))
                        addData['totalMiss'].append(float(0))
                    else:
                        addData['totalAcc'].append(float(totalCorrect/totalSum))
                        addData['unknownRate'].append(float(totalUnknown/totalSum))
                        addData['incorrectRate'].append(float((totalSum - totalUnknown - totalCorrect)/totalSum))
                        addData['knownAcc'].append(float(totalCorrect/(totalSum - totalUnknown)))
                        addData['totalCorrect'].append(totalCorrect)
                        addData['totalUnknown'].append(totalUnknown)
                        addData['totalSum'].append(totalSum)
                        addData['totalMiss'].append(totalSum-totalCorrect-totalUnknown)
    
                    #l = ""
                    #for key in addData.keys():
                    #    l += ",{0}".format(addData[key][line])
                    #print(l)
                
                    line += 1
    
                    #break
                #break
            #break
        #break

summaryDF = pd.DataFrame(addData)

#summaryDF    

In [ ]:
summaryDF.columns

In [ ]:
summaryDF[summaryDF['detectThreshold'] == 0.6]

In [ ]:
summaryDF[summaryDF['totalAcc'] > 0.32]

In [ ]:
ks

In [ ]:
import matplotlib.pyplot as plt

# ax=ax と指定することで、同じグラフ上に描画を指示します
#summaryDF[(summaryDF['dbThreshold'] == 0.95) & (summaryDF['detectThreshold'] == 0.5) & (summaryDF['refine'] == '0.01')].plot(x='k', y=['total'], ax=ax, label=['0.01'])
skipLabel = 99
graphCols = ['totalAcc', 'unknownRate', 'incorrectRate', 'knownAcc'];
graphLabels = ['Total Accuracy (Recall)', 'Unknown Rate', 'Incorrect Rate (Miss)', 'Known Accuracy (Precision)'];
graphStyles = ['C0o-', 'C1s-', 'C2v-', 'C3*--'];
displayCols = ['detectThreshold', 'totalAcc', 'unknownRate', 'incorrectRate', 'knownAcc', 'totalCorrect', 'totalUnknown', 'totalMiss', 'totalSum' ]
for dbThreshold in [0.95, 0.98]:
    for k in [1]:
        #for refine in refines:
        for refine in ['none']:
            fig, ax = plt.subplots(figsize=(10, 6))
            tmpDF = summaryDF[(summaryDF['k'] == k) & (summaryDF['dbThreshold'] == dbThreshold) & (summaryDF['refine'] == str(refine)) & (summaryDF['skipLabel'] == skipLabel)] 
            tmpDF.plot(x='detectThreshold', y=graphCols, ax=ax, label=graphLabels, style=graphStyles)
            #for col,lab,sty in zip(graphCols, graphLabels, graphStyles):
                #tmpDF.plot(x='detectThreshold', y=[col], ax=ax, label=[lab], style=[sty])
                #display(summaryDF[(summaryDF['k'] == k) & (summaryDF['dbThreshold'] == 0.95) & (summaryDF['refine'] == str(refine)) & (summaryDF['skipLabel'] == skipLabel)])
            display(tmpDF[displayCols])
            csv_string = tmpDF[displayCols].to_csv(index=False)
            print(csv_string)
            title = 'Re-kNN CIFAR-10 Performance vs Detection Threshold. Similarity={0:0.2} Refine={1}'.format(dbThreshold, refine)
            ax.set_title(title)
            pngfile = 'Re-kNN_CIFAR10_Performance_vs_Detection_Threshold_Similarity={0:0.2}_Refine={1}.png'.format(dbThreshold, refine)
            plt.savefig(pngfile)
            plt.show()
        


In [ ]:
import matplotlib.pyplot as plt

# ax=ax と指定することで、同じグラフ上に描画を指示します
#summaryDF[(summaryDF['dbThreshold'] == 0.95) & (summaryDF['detectThreshold'] == 0.5) & (summaryDF['refine'] == '0.01')].plot(x='k', y=['total'], ax=ax, label=['0.01'])
skipLabel = 99
graphCols = ['totalAcc', 'unknownRate'];
graphLabels = ['Total Accuracy (Recall)', 'Unknown Rate'];
graphStyles = ['C0', 'C1'];
displayCols = ['refine', 'detectThreshold', 'totalAcc', 'unknownRate', 'incorrectRate', 'knownAcc', 'totalCorrect', 'totalUnknown', 'totalMiss', 'totalSum' ]
for dbThreshold in [0.95, 0.98]:
    for k in [1]:
        fig, ax = plt.subplots(figsize=(10, 6))
    
        #for refine in refines:
        for refine, refileLabel, refineStyle in zip(['none', '0.01'], ['Pre-Refine', 'Post-Refine'], ['o-', 's--']):
            tmpDF = summaryDF[(summaryDF['k'] == k) & (summaryDF['dbThreshold'] == 0.95) & (summaryDF['refine'] == str(refine)) & (summaryDF['skipLabel'] == skipLabel)] 
            csv_string = tmpDF[displayCols].to_csv(index=False)
            print(csv_string)
            for col,colLabel,colStyle in zip(graphCols, graphLabels, graphStyles):
                tmpDF.plot(x='detectThreshold', y=[col], ax=ax, label=["{0} ({1})".format(refileLabel, colLabel)], style=[colStyle+refineStyle])
    
        title = 'Re-kNN CIFAR-10 Performance Pre-Refine vs Post-Refine (Similarity={0:0.2})'.format(dbThreshold)
        ax.set_title(title)
        pngfile = 'Re-kNN_CIFAR10_Performance_Pre-Refine_vs_Post-Refine_(Similarity={0:0.2}).png'.format(dbThreshold)
        plt.savefig(pngfile)
        plt.show()
        


In [ ]:
import matplotlib.pyplot as plt

# ax=ax と指定することで、同じグラフ上に描画を指示します
#summaryDF[(summaryDF['dbThreshold'] == 0.95) & (summaryDF['detectThreshold'] == 0.5) & (summaryDF['refine'] == '0.01')].plot(x='k', y=['total'], ax=ax, label=['0.01'])
skipLabel = 99
graphCols = ['totalAcc', 'unknownRate'];
graphLabels = ['Total Accuracy (Recall)', 'Unknown Rate'];
graphStyles = ['C0', 'C1'];
displayCols = ['refine', 'detectThreshold', 'totalAcc', 'unknownRate', 'incorrectRate', 'knownAcc', 'totalCorrect', 'totalUnknown', 'totalMiss', 'totalSum' ]
for dbThreshold in [0.95, 0.98]:
    for k in [1]:
        fig, ax = plt.subplots(figsize=(10, 6))
    
        #for refine in refines:
        for refine, refileLabel, refineStyle in zip(['none', '0.01'], ['Pre-Refine', 'Post-Refine'], ['o-', 's--']):
            tmpDF = summaryDF[(summaryDF['k'] == k) & (summaryDF['dbThreshold'] == 0.95) & (summaryDF['refine'] == str(refine)) & (summaryDF['skipLabel'] == skipLabel)] 
            csv_string = tmpDF[displayCols].to_csv(index=False)
            print(csv_string)
            for col,colLabel,colStyle in zip(graphCols, graphLabels, graphStyles):
                tmpDF.plot(x='detectThreshold', y=[col], ax=ax, label=["{0} ({1})".format(refileLabel, colLabel)], style=[colStyle+refineStyle])
    
        title = 'Re-kNN CIFAR-10 Performance Pre-Refine vs Post-Refine (Similarity={0:0.2})'.format(dbThreshold)
        ax.set_title(title)
        pngfile = 'Re-kNN_CIFAR10_Performance_Pre-Refine_vs_Post-Refine_(Similarity={0:0.2}).png'.format(dbThreshold)
        plt.savefig(pngfile)
        plt.show()
        


In [ ]:
skipLabels = [0,1,2,3,4,5,6,7,8,9,99]
allLabels = [0,1,2,3,4,5,6,7,8,9]

pairs = [
    [0, 'Label_0', 'Label_0_count', 'Label_0_total', 'Label_0_unknown', 'Label_0_unknown_Acc', 'Label_0_Acc'],
    [1, 'Label_1', 'Label_1_count', 'Label_1_total', 'Label_1_unknown', 'Label_1_unknown_Acc', 'Label_1_Acc'],
    [2, 'Label_2', 'Label_2_count', 'Label_2_total', 'Label_2_unknown', 'Label_2_unknown_Acc', 'Label_2_Acc'],
    [3, 'Label_3', 'Label_3_count', 'Label_3_total', 'Label_3_unknown', 'Label_3_unknown_Acc', 'Label_3_Acc'],
    [4, 'Label_4', 'Label_4_count', 'Label_4_total', 'Label_4_unknown', 'Label_4_unknown_Acc', 'Label_4_Acc'],
    [5, 'Label_5', 'Label_5_count', 'Label_5_total', 'Label_5_unknown', 'Label_5_unknown_Acc', 'Label_5_Acc'],
    [6, 'Label_6', 'Label_6_count', 'Label_6_total', 'Label_6_unknown', 'Label_6_unknown_Acc', 'Label_6_Acc'],
    [7, 'Label_7', 'Label_7_count', 'Label_7_total', 'Label_7_unknown', 'Label_7_unknown_Acc', 'Label_7_Acc'],
    [8, 'Label_8', 'Label_8_count', 'Label_8_total', 'Label_8_unknown', 'Label_8_unknown_Acc', 'Label_8_Acc'],
    [9, 'Label_9', 'Label_9_count', 'Label_9_total', 'Label_9_unknown', 'Label_9_unknown_Acc', 'Label_9_Acc'],
]


similarity = 0.95
detectThreshold = 0.7
k = 3

tmpDf = summaryDF[(summaryDF['dbThreshold'] == similarity) & (summaryDF['detectThreshold'] == detectThreshold) & (summaryDF['refine'] == 'none') & (summaryDF['k'] == k)]

sumDf = pd.DataFrame()
sumDfData = {}
sumDfData['skipLabel'] = []
sumDfData['total'] = []
sumDfData['correct'] = []
sumDfData['unknown'] = []
sumDfData['miss'] = []
sumDfData['correctRate'] = []
sumDfData['unknownRate'] = []
sumDfData['missRate'] = []
sumDfData['accWithoutUnknown'] = []
sumDfData['skipTotal'] = []
sumDfData['skipUnknown'] = []
sumDfData['skipMiss'] = []
sumDfData['skipUnknownRate'] = []
sumDfData['skipMissRate'] = []
sumDfData['withoutSkipTotal'] = []
sumDfData['withoutSkipCorrect'] = []
sumDfData['withoutSkipUnknown'] = []
sumDfData['withoutSkipMiss'] = []
sumDfData['withoutSkipCorrectRate'] = []
sumDfData['withoutSkipUnknownRate'] = []
sumDfData['withoutSkipMissRate'] = []
sumDfData['withoutSkipAccWithoutUnknown'] = []

for skipLabel in skipLabels:
    tmpDf2 = tmpDf[tmpDf['skipLabel']==skipLabel]
    if len(tmpDf2) != 1:
        print("BUG!!")
    sumDfData['skipLabel'].append(skipLabel)
    sumDfData['total'].append(tmpDf2['totalSum'].iloc[0])
    sumDfData['correct'].append(tmpDf2['totalCorrect'].iloc[0])
    sumDfData['unknown'].append(tmpDf2['totalUnknown'].iloc[0])
    sumDfData['miss'].append(tmpDf2['totalSum'].iloc[0] - tmpDf2['totalCorrect'].iloc[0] - tmpDf2['totalUnknown'].iloc[0])
    sumDfData['correctRate'].append(float(tmpDf2['totalCorrect'].iloc[0])/float(tmpDf2['totalSum'].iloc[0]))
    sumDfData['unknownRate'].append(float(tmpDf2['totalUnknown'].iloc[0])/float(tmpDf2['totalSum'].iloc[0]))
    sumDfData['missRate'].append(float(tmpDf2['totalSum'].iloc[0] - tmpDf2['totalCorrect'].iloc[0] - tmpDf2['totalUnknown'].iloc[0])/float(tmpDf2['totalSum'].iloc[0]))
    sumDfData['accWithoutUnknown'].append(float(tmpDf2['totalCorrect'].iloc[0])/float(tmpDf2['totalSum'].iloc[0] - tmpDf2['totalUnknown'].iloc[0]))

    if skipLabel != 99:
        sumDfData['skipTotal'].append(int(tmpDf2[pairs[skipLabel][3]].iloc[0]))
        sumDfData['skipUnknown'].append(int(tmpDf2[pairs[skipLabel][4]].iloc[0]))
        sumDfData['skipMiss'].append(int(tmpDf2[pairs[skipLabel][3]].iloc[0] - tmpDf2[pairs[skipLabel][4]].iloc[0]))
        sumDfData['skipUnknownRate'].append(float(tmpDf2[pairs[skipLabel][4]].iloc[0])/float(tmpDf2[pairs[skipLabel][3]].iloc[0]))
        sumDfData['skipMissRate'].append(float(tmpDf2[pairs[skipLabel][3]].iloc[0] - tmpDf2[pairs[skipLabel][4]].iloc[0])/float(tmpDf2[pairs[skipLabel][3]].iloc[0]))
    else:
        sumDfData['skipTotal'].append(0)
        sumDfData['skipUnknown'].append(0)
        sumDfData['skipMiss'].append(0)
        sumDfData['skipUnknownRate'].append(0)
        sumDfData['skipMissRate'].append(0)

    if skipLabel == 99:
        sumDfData['withoutSkipTotal'].append(tmpDf2['totalSum'].iloc[0])
        sumDfData['withoutSkipCorrect'].append(tmpDf2['totalCorrect'].iloc[0])
        sumDfData['withoutSkipUnknown'].append(tmpDf2['totalUnknown'].iloc[0])
        sumDfData['withoutSkipMiss'].append(tmpDf2['totalSum'].iloc[0] - tmpDf2['totalCorrect'].iloc[0] - tmpDf2['totalUnknown'].iloc[0])
        sumDfData['withoutSkipCorrectRate'].append(float(tmpDf2['totalCorrect'].iloc[0])/float(tmpDf2['totalSum'].iloc[0]))
        sumDfData['withoutSkipUnknownRate'].append(float(tmpDf2['totalUnknown'].iloc[0])/float(tmpDf2['totalSum'].iloc[0]))
        sumDfData['withoutSkipMissRate'].append(float(tmpDf2['totalSum'].iloc[0] - tmpDf2['totalCorrect'].iloc[0] - tmpDf2['totalUnknown'].iloc[0])/float(tmpDf2['totalSum'].iloc[0]))
        sumDfData['withoutSkipAccWithoutUnknown'].append(float(tmpDf2['totalCorrect'].iloc[0])/float(tmpDf2['totalSum'].iloc[0] - tmpDf2['totalUnknown'].iloc[0]))
    else:
        total = 0
        correct = 0
        unknown = 0
        miss = 0
        for chkLabel in allLabels:
            if chkLabel == skipLabel:
                continue
            total += int(tmpDf2[pairs[chkLabel][3]].iloc[0])
            correct += int(tmpDf2[pairs[chkLabel][2]].iloc[0])
            unknown += int(tmpDf2[pairs[chkLabel][4]].iloc[0])
            miss += int(tmpDf2[pairs[chkLabel][3]].iloc[0] - tmpDf2[pairs[chkLabel][2]].iloc[0] - tmpDf2[pairs[chkLabel][4]].iloc[0])
    
        sumDfData['withoutSkipTotal'].append(total)
        sumDfData['withoutSkipCorrect'].append(correct)
        sumDfData['withoutSkipUnknown'].append(unknown)
        sumDfData['withoutSkipMiss'].append(miss)
        sumDfData['withoutSkipCorrectRate'].append(correct/total)
        sumDfData['withoutSkipUnknownRate'].append(unknown/total)
        sumDfData['withoutSkipMissRate'].append(miss/total)
        sumDfData['withoutSkipAccWithoutUnknown'].append(correct/(total-unknown))
    
    #for label in allLabels:
        
sumDf = pd.DataFrame(sumDfData)

In [ ]:
tmpDf

In [ ]:
csv_string = sumDf[['correctRate', 'unknownRate', 'missRate', 'correct', 'unknown', 'miss', 'total', 'skipLabel']].to_csv(index=False)
print(csv_string)

In [ ]:
# 棒グラフを描画
ax = sumDf[['correctRate', 'accWithoutUnknown', 'skipLabel']].set_index('skipLabel').plot(kind='bar')
ax.legend(['Accuracy(total)', 'Accuracy(exclude unknown)'], loc=('lower left'))
plt.ylabel('Score(%)')

title = 'Total Score / include excluded label(Accuracy)'.format(dbThreshold)
ax.set_title(title)
pngfile = 'CIFAR10_TotalScore_includeexcludedlabel(Accuracy).png'.format(dbThreshold)
plt.savefig(pngfile)
plt.show()

In [ ]:
# 棒グラフを描画
ax = sumDf[['unknownRate', 'missRate', 'skipLabel']].set_index('skipLabel').plot(kind='bar')
ax.legend(['Unknwon', 'Miss'])
plt.ylabel('Score(%)')

title = 'Total Score / include excluded label (unknown,miss)'.format(dbThreshold)
ax.set_title(title)
pngfile = 'CIFAR10_TotalScore_includeexcludedlabel(unknown,miss).png'.format(dbThreshold)
plt.savefig(pngfile)

plt.show()

In [ ]:
csv_string = sumDf[['withoutSkipCorrectRate', 'withoutSkipAccWithoutUnknown', 'withoutSkipUnknownRate', 'withoutSkipMissRate', 'withoutSkipTotal','withoutSkipCorrect','withoutSkipUnknown','withoutSkipMiss', 'skipLabel']].to_csv(index=False)
print(csv_string)



In [ ]:
# 棒グラフを描画
ax = sumDf[['withoutSkipCorrectRate', 'withoutSkipAccWithoutUnknown', 'skipLabel']].set_index('skipLabel').plot(kind='bar')
ax.legend(['Accuracy', 'AccuracyWithoutUnknown'])

title = 'Total Score / without excluded label (Accuracy)'.format(dbThreshold)
ax.set_title(title)
pngfile = 'CIFAR10_TotalScore_withoutexcludedlabel(Accuracy).png'.format(dbThreshold)
plt.savefig(pngfile)

plt.ylabel('Score(%)')
plt.show()

In [ ]:
# 棒グラフを描画
ax = sumDf[['withoutSkipUnknownRate', 'withoutSkipMissRate', 'skipLabel']].set_index('skipLabel').plot(kind='bar')
ax.legend(['UnknwonRate', 'MissRate'])

title = 'Total Score / without excluded label (Unknown, miss)'.format(dbThreshold)
ax.set_title(title)
pngfile = 'CIFAR10_TotalScore_withoutexcludedlabel(Unknown,miss).png'.format(dbThreshold)
plt.savefig(pngfile)

plt.ylabel('Score(%)')
plt.show()

In [ ]:
# 棒グラフを描画
ax = sumDf[['unknownRate', 'missRate', 'skipLabel']].set_index('skipLabel').plot(kind='bar')
ax.legend(['UnknwonRate', 'MissRate'])

plt.title('Total Score / include excluded label')
plt.ylabel('Score(%)')
plt.show()

In [ ]:
# 棒グラフを描画
ax = sumDf[['skipUnknownRate', 'skipMissRate', 'skipLabel']].set_index('skipLabel').plot(kind='bar')
ax.legend(['Unknwon', 'Miss'])

plt.title('Unknown Label Score (by Rate)')
plt.ylabel('Rate')
plt.show()

In [ ]:
summaryDF.columns

In [ ]:
csv_string = sumDf[['skipUnknownRate', 'skipMissRate', 'skipUnknown', 'skipMiss', 'skipLabel']].to_csv(index=False)
print(csv_string)

In [ ]:
# 棒グラフを描画
ax = sumDf[['skipUnknown', 'skipMiss', 'skipLabel']][sumDf['skipLabel'] != 99].set_index('skipLabel').plot(kind='bar')
ax.legend(['Unknwon', 'Miss'])

plt.ylabel('Count')

title = 'Unknown Label Score (by Count)'.format(dbThreshold)
ax.set_title(title)
pngfile = 'CIFAR10_UnknownLabelScore(byCount).png'.format(dbThreshold)
plt.savefig(pngfile)

plt.show()

In [ ]:
# 棒グラフを描画
ax = sumDf[['skipUnknownRate', 'skipMissRate', 'skipLabel']][sumDf['skipLabel'] != 99].set_index('skipLabel').plot(kind='bar')
ax.legend(['Unknwon', 'Miss'])
plt.ylabel('Rate')

title = 'Unknown Label Score (by Rate)'.format(dbThreshold)
ax.set_title(title)
pngfile = 'CIFAR10_UnknownLabelScore(byRate).png'.format(dbThreshold)
plt.savefig(pngfile)

plt.show()

In [ ]:
# 棒グラフを描画
sumDf[['correct', 'unknown', 'miss', 'skipLabel']].set_index('skipLabel').plot(kind='bar')

plt.title('Sales and Profit by Store')
plt.ylabel('Value')
plt.show()

In [ ]:
# ラベル単位での正答数をカウント
pairs = [
    [' 0', 'Label_0', 'Label_0_count', 'Label_0_total', 'Label_0_unknown', 'Label_0_unknown_Acc', 'Label_0_Acc'],
    ['1', 'Label_1', 'Label_1_count', 'Label_1_total', 'Label_1_unknown', 'Label_1_unknown_Acc', 'Label_1_Acc'],
    ['2', 'Label_2', 'Label_2_count', 'Label_2_total', 'Label_2_unknown', 'Label_2_unknown_Acc', 'Label_2_Acc'],
    ['3', 'Label_3', 'Label_3_count', 'Label_3_total', 'Label_3_unknown', 'Label_3_unknown_Acc', 'Label_3_Acc'],
    ['4', 'Label_4', 'Label_4_count', 'Label_4_total', 'Label_4_unknown', 'Label_4_unknown_Acc', 'Label_4_Acc'],
    ['5', 'Label_5', 'Label_5_count', 'Label_5_total', 'Label_5_unknown', 'Label_5_unknown_Acc', 'Label_5_Acc'],
    ['6', 'Label_6', 'Label_6_count', 'Label_6_total', 'Label_6_unknown', 'Label_6_unknown_Acc', 'Label_6_Acc'],
    ['7', 'Label_7', 'Label_7_count', 'Label_7_total', 'Label_7_unknown', 'Label_7_unknown_Acc', 'Label_7_Acc'],
    ['8', 'Label_8', 'Label_8_count', 'Label_8_total', 'Label_8_unknown', 'Label_8_unknown_Acc', 'Label_8_Acc'],
    ['9', 'Label_9', 'Label_9_count', 'Label_9_total', 'Label_9_unknown', 'Label_9_unknown_Acc', 'Label_9_Acc'],
]

skipLabels = [0,1,2,3,4,5,6,7,8,9,99]
allLabels = [0,1,2,3,4,5,6,7,8,9]

#summaryDF = pd.DataFrame()
#summaryDF = pd.DataFrame(columns=['accuracy'])

indexes = []
addData = {}

ks = df[' k'].unique()
dbThresholds = df[' dbThreshold'].unique()
detectThresholds = df[' detectThreshold'].unique();
refines = df['refine'].unique()

line = 0

for skipLabel in [99]:

    #for k in ks:
    for k in [3]:
    
        #for dbThreshold in dbThresholds:
        for dbThreshold in [0.95]:
    
            for detectThreshold in detectThresholds:
    
                #for refine in refines:
                for refine in ['none']:

                    df_tmp = df[(df[' k']==k) & (df[' dbThreshold'] == dbThreshold) & (df[' detectThreshold'] == detectThreshold) & (df['refine'] == refine) & (df[' skipLabel'] == skipLabel)]
                    #display(df_tmp)

                    print(detectThreshold)
                    
                    mtx = {}
                    for l1 in allLabels:
                        mtx[l1] = {}
                        for l2 in skipLabels:
                            mtx[l1][l2] = 0
        
                    for l1 in allLabels:
                        df_tmp1 = df_tmp[df_tmp[' predictLabels'] == pairs[l1][1]]
                        #display(df_tmp1)

                        for l2 in allLabels:
                            if len(df_tmp1[pairs[l2][0]]) != 1:
                                print("Error")
                                display(df_tmp1[pairs[l2][0]])
                                break;
                            mtx[l1][l2] = int(df_tmp1[pairs[l2][0]].iloc[0])

                        if len(df_tmp1[' Unknown']) != 1:
                            print("Error")
                            display(df_tmp1[' Unknown'])
                            break;
                        mtx[l1][99] = int(df_tmp1[' Unknown'].iloc[0])

                    # confusion matrix ができた
                    print("label,0,1,2,3,4,5,6,7,8,9,unknown")
                    for clabel in allLabels:
                        line = '| {0}'.format(clabel)
                        for plabel in skipLabels:
                            line += ' | {0}'.format(mtx[clabel][plabel])
                        print(line + " |")
                    


In [ ]:
df_tmp1[pairs[l2][0]]